In [7]:
import sys
print("Python version:", sys.version)
print("\nChecking all required packages:\n")
packages = {
    'fastapi': 'FastAPI',
    'tweepy': 'Tweepy',
    'textblob': 'TextBlob',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'uvicorn': 'uvicorn',
    'jupyter': 'Jupyter'
}
for module, name in packages.items():
    try:
        mod = __import__(module)
        version = getattr(mod, '__version__', 'installed')
        print(f"{name}: {version}")
    except ImportError:
        print(f"{name}: NOT installed")

Python version: 3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]

Checking all required packages:

FastAPI: 0.128.7
Tweepy: 4.16.0
TextBlob: installed
scikit-learn: 1.8.0
pandas: 3.0.0
uvicorn: 0.40.0
Jupyter: installed


In [8]:
from textblob import TextBlob

def analyze_sentiment(text):
    try:
        blob = TextBlob(text)
        polarity = blob.sentiment.polarity
        if polarity > 0.1:
            sentiment_category = "Positive"
        elif polarity < -0.1:
            sentiment_category = "Negative"
        else:
            sentiment_category = "Neutral"
        return {
            "text": text,
            "polarity": round(polarity, 4),
            "sentiment": sentiment_category
            }
    except Exception as e:
        return {"text": text, "error": str(e)}
print("✓ Sentiment analysis function created")
test_texts = [
    "I love this!", 
    "This is awful", 
    "The sky is blue" 
    ]
print("\nTesting sentiment analysis:\n")
for text in test_texts:
    result = analyze_sentiment(text)
    print(f"Text: '{result['text']}'")
    print(f" Polarity: {result['polarity']}")
    print(f" Sentiment: {result['sentiment']}")
    print()

✓ Sentiment analysis function created

Testing sentiment analysis:

Text: 'I love this!'
 Polarity: 0.625
 Sentiment: Positive

Text: 'This is awful'
 Polarity: -1.0
 Sentiment: Negative

Text: 'The sky is blue'
 Polarity: 0.0
 Sentiment: Neutral



In [9]:
import pandas as pd
sample_tweets = [
    {"text": "I absolutely love this new product! It's amazing!", "author":"user1"},
    {"text": "This is terrible. Worst experience ever.", "author": "user2"},
    {"text": "The weather is nice today.", "author": "user3"},
    {"text": "I'm so happy with my purchase! Highly recommend!", "author":"user4"},
    {"text": "This app keeps crashing. Very disappointed.", "author": "user5"},
    {"text": "Just had the best coffee of my life!", "author": "user6"},
    {"text": "The service was slow and food was cold.", "author": "user7"},
    {"text": "Learning Python is fun and rewarding.", "author": "user8"},
    {"text": "I hate waiting in long queues.", "author": "user9"},
    {"text": "The sunset today was beautiful.", "author": "user10"},
    {"text": "This movie is absolutely brilliant!", "author": "user11"},
    {"text": "I'm frustrated with the lack of updates.", "author": "user12"},
    ]
tweets_df = pd.DataFrame(sample_tweets)
print("✓ Sample tweets dataset created")
print(f"Total tweets: {len(tweets_df)}")
print("\nSample data:")
print(tweets_df.head())

✓ Sample tweets dataset created
Total tweets: 12

Sample data:
                                                text author
0  I absolutely love this new product! It's amazing!  user1
1           This is terrible. Worst experience ever.  user2
2                         The weather is nice today.  user3
3   I'm so happy with my purchase! Highly recommend!  user4
4        This app keeps crashing. Very disappointed.  user5


In [10]:
sentiment_results = []

for idx, row in tweets_df.iterrows():
    sentiment_data = analyze_sentiment(row['text'])
    sentiment_data['author'] = row['author']
    sentiment_results.append(sentiment_data)

results_df = pd.DataFrame(sentiment_results)

print("Sentiment analysis completed")
print(f"Total tweets analyzed: {len(results_df)}\n")

sentiment_counts = results_df['sentiment'].value_counts()
print("Sentiment Distribution:")
for sentiment, count in sentiment_counts.items():
    percentage = (count / len(results_df)) * 100
    print(f" {sentiment}: {count} ({percentage:.1f}%)")
print(f"\nAverage Polarity: {results_df['polarity'].mean():.4f}")
print("\nDetailed Results:")
print(results_df[['text', 'author', 'sentiment', 'polarity']])

Sentiment analysis completed
Total tweets analyzed: 12

Sentiment Distribution:
 Positive: 7 (58.3%)
 Negative: 5 (41.7%)

Average Polarity: 0.1145

Detailed Results:
                                                 text  author sentiment  \
0   I absolutely love this new product! It's amazing!   user1  Positive   
1            This is terrible. Worst experience ever.   user2  Negative   
2                          The weather is nice today.   user3  Positive   
3    I'm so happy with my purchase! Highly recommend!   user4  Positive   
4         This app keeps crashing. Very disappointed.   user5  Negative   
5                Just had the best coffee of my life!   user6  Positive   
6             The service was slow and food was cold.   user7  Negative   
7               Learning Python is fun and rewarding.   user8  Positive   
8                      I hate waiting in long queues.   user9  Negative   
9                     The sunset today was beautiful.  user10  Positive   
10      

In [11]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(
    title="Sentiment Prediction API",
    description="Analyzes sentiment of tweets using TextBlob",
    version="1.0.0",
    swagger_ui_parameters={"defaultModelsExpandDepth": -1}
)

class TweetInput(BaseModel):
    text: str
    author: Optional[str] = "Anonymous"
class SentimentResult(BaseModel):
    text: str
    author: str
    sentiment: str
    polarity: float
class BulkAnalysisResponse(BaseModel):
    total_tweets: int
    results: List[SentimentResult]
    sentiment_distribution: dict
print("FastAPI application initialized")
print(f"Title: {app.title}")
print(f"Version: {app.version}")
print("\nPydantic models created:")
print(" - TweetInput (for requests)")
print(" - SentimentResult (for individual results)")
print(" - BulkAnalysisResponse (for complete response)")

FastAPI application initialized
Title: Sentiment Prediction API
Version: 1.0.0

Pydantic models created:
 - TweetInput (for requests)
 - SentimentResult (for individual results)
 - BulkAnalysisResponse (for complete response)


In [12]:
@app.post("/analyze_tweets/", response_model=BulkAnalysisResponse)
def analyze_tweets(tweets_input: List[TweetInput]):
    results = []
    for tweet in tweets_input:
        sentiment_data = analyze_sentiment(tweet.text)
        result = SentimentResult(
            text=sentiment_data['text'],
            author=tweet.author,
            sentiment=sentiment_data['sentiment'],
            polarity=sentiment_data['polarity']
        )
        results.append(result)
    sentiments = [r.sentiment for r in results]
    sentiment_distribution = {
        "Positive": sentiments.count("Positive"),
        "Negative": sentiments.count("Negative"),
        "Neutral": sentiments.count("Neutral")
        }
    return BulkAnalysisResponse(
        total_tweets=len(results),
        results=results,
        sentiment_distribution=sentiment_distribution
    )
print("API endpoint created: POST /analyze_tweets/")
print("\nEndpoint details:")
print(" Path: /analyze_tweets/")
print(" Method: POST")
print(" Input: List of tweets with text and author")
print(" Output: Analyzed sentiments with distribution")

API endpoint created: POST /analyze_tweets/

Endpoint details:
 Path: /analyze_tweets/
 Method: POST
 Input: List of tweets with text and author
 Output: Analyzed sentiments with distribution
